# Gruppo 1: Andrea Pezzolla & Matteo Maruca
Progetto 2 DataScience, SUPSI, I2A, 2026

Fonte dataset:
https://opendata.swiss/en/dataset/geschwindigkeitsmonitoring-einzelmessungen-ab-2024

In [ ]:
# Import
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
# Dataframe
df = pd.read_csv("100097.csv", sep=';')

In [ ]:
# TRADUZIONI COLONNE 

# ── Rinomina colonne in italiano ──────────────────────────────
df = df.rename(columns={
    "Timestamp"                         : "timestamp",
    "Messung-ID"                        : "id_postazione",
    "Richtung ID"                       : "id_direzione",
    "Geschwindigkeit"                   : "velocita_kmh",
    "Zeit"                              : "ora",
    "Datum"                             : "data",
    "Datum und Zeit"                    : "data_ora",
    "Messbeginn"                        : "inizio_misurazione",
    "Messende"                          : "fine_misurazione",
    "Zone"                              : "zona_limite",
    "Ort"                               : "localita",
    "Richtung"                          : "direzione",
    "Koordinaten"                       : "coordinate",
    "Übertretungsquote"                 : "tasso_infrazioni_pct",
    "Geschwindigkeit V50"               : "v50",
    "Geschwindigkeit V85"               : "v85",
    "Strasse"                           : "via",
    "Hausnummer"                        : "numero_civico",
    "Fahrzeuge"                         : "n_veicoli",
    "Fahrzeuglänge"                     : "lunghezza_veicolo_m",
    "Kennzahlen pro Mess-Standort"      : "link_statistiche",
    "geo_point_2d"                      : "geo_point",
})

# ── Conversioni di tipo ───────────────────────────────────────
df["timestamp"]           = pd.to_datetime(df["timestamp"], utc=True)
df["inizio_misurazione"]  = pd.to_datetime(df["inizio_misurazione"])
df["fine_misurazione"]    = pd.to_datetime(df["fine_misurazione"])

# Estrai colonne utili da timestamp
df["anno"]         = df["timestamp"].dt.year
df["mese"]         = df["timestamp"].dt.month
df["giorno_sett"]  = df["timestamp"].dt.day_name(locale="it_IT.UTF-8")  # es. "Lunedì"
df["ora_intera"]   = df["timestamp"].dt.hour

# Zona limite come categoria
#df["zona_limite"] = df["zona_limite"].astype(int).astype(str).apply(lambda x: f"Zona {x}")

print("Colonne rinominate:")
print(df.columns.tolist())
print(f"\nDimensioni: {df.shape[0]:,} righe × {df.shape[1]} colonne")
print(df.dtypes)

In [ ]:
df["zona_limite"].unique().max()

# Top 15 postazioni per tasso medio di infrazioni

In [ ]:
# Top 15 postazioni per tasso medio di infrazioni

# Aggiungiamo 'zona_limite' al groupby per conservare il dato

top_infrazioni1 = (
    df.groupby(["localita", "via", "zona_limite"])["tasso_infrazioni_pct"]
    .mean()
    .reset_index()
    .sort_values("tasso_infrazioni_pct", ascending=False)
    .head(15)
)

fig2 = px.bar(
    top_infrazioni1,
    x="tasso_infrazioni_pct",
    y="via",
    orientation="h",
    color="tasso_infrazioni_pct",
    color_continuous_scale="Teal", 
    title="<b>Top 15 Postazioni per Tasso di Infrazioni</b>",
    labels={
        "tasso_infrazioni_pct": "Tasso infrazioni (%)",
        "via":                  "Via",
        "localita":             "Località",
        "zona_limite":          "Limite (km/h)"
    },
    # Configurazione dettagliata dell'hover
    hover_data={
        "via": False, # Nasconde la via dal box perché è già sull'asse Y
        "localita": True,
        "zona_limite": True,
        "tasso_infrazioni_pct": ":.2f" # Formatta il numero con 2 decimali (es. 12.34 al posto di 12.34567)
    }
)

fig2.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False,
    plot_bgcolor="white", # Sfondo bianco per una lettura più pulita
)

# Aggiungiamo la griglia verticale leggera per facilitare la lettura dei valori
fig2.update_xaxes(showgrid=True, gridcolor="#E5E5E5")

fig2.show()


# Top 15 postazioni per tasso di infrazione (calcolato manualmente)
calcolato manualmente senza usare la colonna gia presente nel dataset

In [ ]:
# Top 15 postazioni per tasso medio di infrazioni

# Aggiungiamo 'zona_limite' al groupby per conservare il dato

df['superamento'] = df['velocita_kmh'] > df['zona_limite']

#raggruppiamo per ogni via localita e zona limite e per ognuna di esse facciamo la media del superamento che è vero o falso (da la proporzione)
radar_stats = (
    df.groupby(["localita", "via", "zona_limite"])['superamento']
    .agg(['mean', 'count'])
    .reset_index()
)

#Filtriamo quei radar con almeno 1000 registrazioni
radar_stats = radar_stats[radar_stats['count'] >= 1000]

#trasformiamo in prct
radar_stats['infraz_perc'] = radar_stats['mean']*100

top_infrazioni = (
    radar_stats.sort_values('infraz_perc',ascending=False)
    .head(15)
)

fig2 = px.bar(
    top_infrazioni,
    x="infraz_perc",
    y="via",
    orientation="h",
    color="infraz_perc",
    color_continuous_scale="Teal", 
    title="<b>Top 15 Postazioni per Tasso di Infrazioni</b>",
    labels={
        "infraz_perc": "Tasso infrazioni (%)",
        "via":                  "Via",
        "localita":             "Località",
        "zona_limite":          "Limite (km/h)"
    },
    # Configurazione dettagliata dell'hover
    hover_data={
        "via": False, # Nasconde la via dal box perché è già sull'asse Y
        "localita": True,
        "zona_limite": True,
        "infraz_perc": ":.2f", # Formatta il numero con 2 decimali (es. 12.34 al posto di 12.34567)
        "count": True
    
    }
)

fig2.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False,
    plot_bgcolor="white", # Sfondo bianco per una lettura più pulita
)

# Aggiungiamo la griglia verticale leggera per facilitare la lettura dei valori
fig2.update_xaxes(showgrid=True, gridcolor="#E5E5E5")

fig2.show()


# Distribuzione della velocità sulla postazione con registrazioni peggiori
Il grafico mostra come in media i veicoli che passano per questa strada stiano tra i 45 e 54kmh superando il limite di velocità (40kmh).

In [ ]:
# Zoom sulla peggiore postazione: Distribuzione velocità

# 1. Estraiamo automaticamente il nome della via e il limite dal primo in classifica
via_peggiore = top_infrazioni.iloc[0]["via"]
limite_peggiore = top_infrazioni.iloc[0]["zona_limite"]

# 2. Filtriamo il dataframe originale 'df' per prendere solo i dati di quella via
df_peggiore = df[df["via"] == via_peggiore]

# 3. Creiamo l'istogramma (la "campana") delle velocità misurate
fig3 = px.histogram(
    df_peggiore,
    x="velocita_kmh",
    nbins=30, # Regola questo numero per fare le barre più o meno spesse
    title=f"<b>Distribuzione delle Velocità - {via_peggiore}</b>",
    labels={
        "velocita_kmh": "Velocità Misurata (km/h)",
        "count": "Numero di Veicoli"
    },
    color_discrete_sequence=["#1f77b4"],
    opacity=0.8
)

# 4. Aggiungiamo la riga verticale rossa per indicare dove inizia l'infrazione
fig3.add_vline(
    x=limite_peggiore, 
    line_width=3, 
    line_dash="dash", 
    line_color="red",
    annotation_text=f"Limite ({limite_peggiore} km/h)", 
    annotation_position="top right",
    annotation_font_color="red",
    annotation_font_size=14
)

# 5. Pulizia del layout
fig3.update_layout(
    plot_bgcolor="white",
    bargap=0.05, # Piccolo spazio tra le barre per renderle più leggibili
    yaxis_title="Numero di Veicoli"
)

# Aggiungiamo griglie leggere
fig3.update_xaxes(showgrid=True, gridcolor="#E5E5E5")
fig3.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

fig3.show()

# Numero di misurazioni per giorno e ora (media giornaliera)
Il grafico mostra come il numero di registrazioni medie giornaliere siano più frequenti durante le ore del giorno rispetto a quelle notturne.

In [ ]:
#Heatmap - nr. misurazioni per giorno e ora (media giornaliera)

# Ordine corretto dei giorni
ordine_giorni = ["Lunedì", "Martedì", "Mercoledì", "Giovedì", "Venerdì", "Sabato", "Domenica"]

# 1. Contiamo i veicoli (numero di righe) per ogni SINGOLA DATA e ORA
df_volumi_giornalieri = df.groupby(["data", "giorno_sett", "ora_intera"]).size().reset_index(name="veicoli_orari")

# 2. Calcoliamo la MEDIA di questi volumi per avere il dato tipico "per giorno"
pivot_traffico = (
    df_volumi_giornalieri.groupby(["giorno_sett", "ora_intera"])["veicoli_orari"]
    .mean()
    .reset_index()
    .pivot(index="giorno_sett", columns="ora_intera", values="veicoli_orari")
    .reindex(ordine_giorni)
)

# 3. Creiamo la Heatmap con il nuovo Titolo e Sottotitolo
fig3 = px.imshow(
    pivot_traffico,
    color_continuous_scale="inferno", 
    aspect="auto",
    title="<b>Misurazioni/Giorno per Giorno e Ora</b><br><sup style='font-size:14px; font-weight:normal; color:gray'>Numero medio giornaliero di veicoli rilevati all'ora su singola giornata</sup>",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Rilevazioni"
    }
)

# Aggiornamento dei font e del design generale
fig3.update_layout(
    title=dict(font=dict(size=20)),
    xaxis_title="<b>Ora del giorno</b>",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=80) # Diamo respiro al sottotitolo
)

# Forza la visualizzazione di tutte le ore (da 0 a 23)
fig3.update_xaxes(
    tickmode="linear", 
    tick0=0, 
    dtick=1,
    showgrid=False
)

fig3.update_yaxes(showgrid=False)

# Tooltip personalizzato aggiornato
fig3.update_traces(
    hovertemplate="<b>%{y}</b> alle ore <b>%{x}:00</b><br>Media rilevazioni: %{z:,.0f} veicoli<extra></extra>"
)

fig3.show()

# Tasso di infrazione per giorno e ora (calcolato manualmente)
Il grafico mostra chiaramente che in percentuale il numero di infrazioni avviene maggiormente nelle ore notturne rispetto alle ore del giorno.
(Si tende a superare il limite durante le ore notturne).

In [ ]:
#Tasso Infrazioni per Giorno e Ora - Heatmap
#Metodo 2 con tasso di infrazione calcolato automaticamente

df['superamento'] = df['velocita_kmh'] > df['zona_limite']

df_raggruppato = (
    df.groupby(['giorno_sett', 'ora_intera'])['superamento']
    .mean()
    .reset_index()
)

df_raggruppato['tasso_medio_pct'] = df_raggruppato['superamento'] * 100

ordine_giorni = ["Lunedì", "Martedì", "Mercoledì", "Giovedì", "Venerdì", "Sabato", "Domenica"]

pivot_inf = (
    df_raggruppato.pivot(index="giorno_sett", columns="ora_intera", values="tasso_medio_pct")
    .reindex(ordine_giorni)
)


fig4 = px.imshow(
    pivot_inf,
    color_continuous_scale="inferno",
    range_color=[pivot_inf.min().min(), pivot_inf.max().max()],
    aspect="auto",
    title="<b>Tasso Infrazioni per Giorno e Ora (%)</b><br><sup style='font-size:13px; font-weight:normal; color:gray'>Calcolo del tasso ponderato sui volumi di traffico effettivi</sup>",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Infrazioni (%)"
    }
)

fig4.update_layout(
    title=dict(font=dict(size=20)),
    xaxis_title="<b>Ora del giorno</b>",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

# Forza la visualizzazione di tutte le ore (da 0 a 23)
fig4.update_xaxes(
    tickmode="linear", 
    tick0=0, 
    dtick=1,
    showgrid=False
)

fig4.update_yaxes(showgrid=False)

fig4.update_traces(
    hovertemplate="<b>%{y}</b> alle ore <b>%{x}:00</b><br>Tasso infrazioni: %{z:.2f}%<extra></extra>"
)

fig4.show()


# Media annua infrazioni veicoli

In [ ]:
# SubPlot – media annua infrazioni veicoli

# 1. Categorie veicoli
bins   = [0, 2.5, 5.0, 7.0, 100.0]
labels = ["Moto", "Auto", "Furgoni", "Camion/Bus"]

df["cat_veicolo"] = pd.cut(df["lunghezza_veicolo_m"], bins=bins, labels=labels, right=True)

# Calcolo anni totali per la media annua
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
anni_totali = df["timestamp"].dt.year.nunique()
if anni_totali == 0: anni_totali = 1

# Pulizia zone 
df_clean = df.dropna(subset=["zona_limite"]).copy()
df_clean["zona_limite_str"] = df_clean["zona_limite"].apply(lambda x: str(int(x)))

ordine_zone = sorted(df_clean["zona_limite"].unique())
ordine_zone_str = [str(int(z)) for z in ordine_zone]

# 2. Calcolo delle infrazioni reali per riga
df_clean["infrazione_reale"] = df_clean["tasso_infrazioni_pct"] / 100

# 3. Raggruppamento e calcolo della MEDIA ANNUA
df_infrazioni = (
    df_clean.groupby(["zona_limite_str", "cat_veicolo"], observed=False)["infrazione_reale"]
    .sum()
    .reset_index(name="Totale_Infrazioni")
)

df_infrazioni["Media_Annua"] = df_infrazioni["Totale_Infrazioni"] / anni_totali

# 4. Calcolo Percentuali
totale_media_zona = df_infrazioni.groupby("zona_limite_str", observed=False)["Media_Annua"].transform("sum")
df_infrazioni["Percentuale"] = (df_infrazioni["Media_Annua"] / totale_media_zona * 100).fillna(0)

# Palette colori
colori_veicoli = {
    "Moto": "#00B4D8",        
    "Auto": "#E07B39",        
    "Furgoni": "#2A9D8F",     
    "Camion/Bus": "#D62828"   
}

# 5. Definizione del SubPlot
fig6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Media Annua Infrazioni", "Distribuzione % delle Infrazioni"),
    shared_yaxes=True,
    column_widths=[0.60, 0.40]
)

# --- GRAFICO A SINISTRA - barre orizzontali raggruppate ---
for cat in reversed(labels): 
    subset = df_infrazioni[df_infrazioni["cat_veicolo"] == cat]
    
    fig6.add_trace(
        go.Bar(
            x=subset["Media_Annua"],
            y=subset["zona_limite_str"],
            name=cat,
            orientation='h',
            marker_color=colori_veicoli[cat],
            legendgroup=cat,
            offsetgroup=cat, 
            hovertemplate='<b>Zona %{y}</b><br>%{x:,.0f} infrazioni/anno<extra>' + cat + '</extra>'
        ),
        row=1, col=1
    )

# --- GRAFICO A DESTRA - barre orizzontali in pila al 100% ---
cumsum = {z: 0.0 for z in ordine_zone_str}

for cat in labels:
    subset = df_infrazioni[df_infrazioni["cat_veicolo"] == cat].set_index("zona_limite_str")

    x_vals    = [subset.loc[z, "Percentuale"] if z in subset.index else 0.0 for z in ordine_zone_str]
    base_vals = [cumsum[z] for z in ordine_zone_str]

    for z, x in zip(ordine_zone_str, x_vals):
        cumsum[z] += x

    fig6.add_trace(
        go.Bar(
            x=x_vals,
            y=ordine_zone_str,
            base=base_vals,             
            name=cat,
            orientation='h',
            marker_color=colori_veicoli[cat],
            legendgroup=cat,
            width=0.75,       
            offset=-0.375,    
            showlegend=False, 
            hovertemplate='<b>Zona %{y}</b><br>%{x:.1f}% del totale<extra>' + cat + '</extra>'
        ),
        row=1, col=2
    )

# 6. Layout finale
fig6.update_layout(
    barmode='group',
    bargap=0.15,          
    bargroupgap=0.02,     
    title="<b>Infrazioni/Anno per Categoria Veicolo e Limite Velocità</b>",
    legend_title="<b>Categoria Veicolo</b>",
    legend_traceorder="reversed",
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=550,
    margin=dict(t=80, b=40, l=60, r=20)
)

# Linee di separazione orizzontali
for i in range(len(ordine_zone_str) - 1):
    fig6.add_hline(
        y=i + 0.5, 
        line_dash="solid",    
        line_color="#E0E0E0", 
        line_width=1.5,
        layer="below",        
        row='all', col='all'  
    )

# Assi X
fig6.update_xaxes(title_text="Infrazioni / Anno", showgrid=True, gridcolor="#E5E5E5", row=1, col=1)
fig6.update_xaxes(title_text="% Infrazioni", range=[0, 100], showgrid=True, gridcolor="#E5E5E5", ticksuffix="%", row=1, col=2)

# Asse Y
fig6.update_yaxes(
    title_text="<b>Zona</b>", 
    categoryorder='array', 
    categoryarray=ordine_zone_str[::-1], 
    row=1, col=1
)

fig6.show()

# Mappa interattiva delle postazioni (Tasso di infrazione VS Volume traffico)
Il grafico mostra la correlazione tra il tasso di infrazione e il volume di traffico totale registrato in ogni radar.

Ci siamo chiesti, ma nelle strade più trafficate si tende a superare più facilmente il limite?

No non è strettamente correlato.

In [ ]:
df[["lat", "lon"]] = df["geo_point"].str.split(",", expand=True).astype(float)

df['superamento'] = df['velocita_kmh'] > df['zona_limite']

mappa = (
    df.groupby(["id_postazione", "via", "zona_limite", "lat", "lon"])
    .agg(
        infrazioni_pct = ("superamento", "mean"),
        tot_veicoli    = ("superamento", "count"),
        vel_media      = ("velocita_kmh", "mean")
    )
    .reset_index()
)

mappa["infrazioni_pct"] = mappa["infrazioni_pct"] * 100

fig8 = px.scatter_map(
    mappa,
    lat="lat",
    lon="lon",
    color="infrazioni_pct",
    size="tot_veicoli",
    size_max=25,
    hover_name="via",
    hover_data={
        "lat": False,
        "lon": False,
        "infrazioni_pct": ":.2f", 
        "vel_media": ":.1f", 
        "zona_limite": True,
        "tot_veicoli": True
    },
    color_continuous_scale="inferno",
    zoom=12,
    map_style="carto-positron",
    title="<b>Mappa Postazioni – Tasso di Infrazioni vs Volume di Traffico</b>",
    labels={
        "infrazioni_pct": "Infrazioni (%)",
        "vel_media":      "Vel. media (km/h)",
        "tot_veicoli":    "Volume di Traffico",
        "zona_limite":    "Limite"
    }
)

fig8.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    title=dict(font=dict(size=20))
)

fig8.show()

# Mappa di attività dei radar
Il grafico mostra un evidente pattern a diagonale, indicando un'attivazione sequenziale dei radar nel tempo. Questo suggerisce che le postazioni non operano simultaneamente, ma vengono spostate o attivate a rotazione mese dopo mese.

In [ ]:
df['anno_mese'] = df['anno'].astype(str) + '-' + df['mese'].astype(str).str.zfill(2)

#df_filtrato = df[(df['id_postazione'] > 750) & (df['id_postazione'] < 825)].copy()

df_raggruppato = df.groupby(['id_postazione', 'anno_mese']).size().reset_index(name='conteggio')

pivot_radar = df_raggruppato.pivot(index="id_postazione", columns="anno_mese", values="conteggio")

pivot_radar = pivot_radar.fillna(0)
pivot_radar = (pivot_radar > 0).astype(int)

fig = px.imshow(
    pivot_radar,
    color_continuous_scale=[[0, "white"], [1, "teal"]],
    aspect="auto",
    title="<b>Mappa delle Attività dei Radar (2024 - 2026)</b><br><sup style='font-size:13px; font-weight:normal; color:gray'>Quadrato colorato = almeno una registrazione nel mese</sup>",
    labels={
        "x": "Periodo (Anno-Mese)",
        "y": "ID Radar",
        "color": "Stato"
    }
)

fig.update_layout(
    coloraxis_showscale=False,
    plot_bgcolor="lightgray",
    paper_bgcolor="white",
    xaxis_title="<b>Mese dell'anno</b>",
    yaxis_title="<b>ID Radar</b>",
)

fig.update_xaxes(type='category', showgrid=True, gridcolor="#E5E5E5")
fig.update_yaxes(type='category', showgrid=True, gridcolor="#E5E5E5")

fig.update_traces(
    hovertemplate="Radar: <b>%{y}</b><br>Mese: <b>%{x}</b><br>Stato: %{z}<extra></extra>"
)

fig.show()